# Rhythm Perception Data Analysis

### How to run
Run all cells in 'Definitions', then run the 'Initialization' cell to build an object named `analysis`, which contains the field `all_data` — the entire processed trial-level dataframe — plus the analysis and plotting methods used below.

The remaining sections either plot figures (saved to `Jupyter Output`) or export `.csv` files that are used by the R scripts for further plotting and statistical analysis.


### Path Requirements
Paths are assumed to be set up like this for the code to run:

- `LOCAL_DIR`
    - `Raw Data/All Birds/` — raw per-bird CSV files
    - `Bird_Summary.xlsx` — subject metadata (name, species, sex, training order, include success/include flags)
    - `AllData.csv` — cache of the raw dataframe
    - `Jupyter Output/` — created automatically; has all figures and exported CSVs
- `STIM_DIR`
    - `Rule_Training_40-275/` — rule-training stimuli, containing `Stimulus Identification.txt`, `File Data/` (timulus header files) and `Stim Files/` (.wav files s)
    - `Training_Probe/` — probe stimuli (same folder structure as above)

Python environment (versions used): pandas 2.2.3, numpy 1.26.4, scipy 1.13.1, matplotlib 3.9.2, openpyxl 3.1.5.

# Definitions

In [1]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
import datetime as dt
import matplotlib as mpl
from matplotlib.ticker import FuncFormatter
import matplotlib.ticker as mticker
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.io import wavfile
import platform

# Data paths
if platform.system() == "Windows":
    LOCAL_DIR = Path(r"")
    STIM_DIR = Path(r"R:\Data\RhythmPerception\Stimuli")
else:
    LOCAL_DIR = Path("/Users/xutianle/Desktop/Kao Lab/Data Local")
    STIM_DIR = Path('/Users/xutianle/Desktop/Research/Stimuli & Audio/Stimuli')

# Ignore divide-by-zero
np.seterr(divide="ignore", invalid="ignore")

# figure style
mpl.rcParams.update({
    "font.family":     "sans-serif",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype":     42,
    "ps.fonttype":      42,
    "svg.fonttype":     "none",
    "mathtext.fontset": "custom",
    "mathtext.rm":      "Arial",
    "mathtext.it":      "Arial:italic",
    "mathtext.bf":      "Arial:bold",
    "font.size":        11,
    "axes.titlesize":   10,
    "axes.labelsize":   8,
    "xtick.labelsize":  8,
    "ytick.labelsize":  8,
    "legend.fontsize":  11,
    "legend.title_fontsize": 11,
    "savefig.dpi":      1200,
    "savefig.bbox":     "standard",
    "legend.frameon":   False,
    "axes.edgecolor":   "black",
    "axes.linewidth":   1,
    "axes.grid":        False,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.facecolor":   "white",
    "figure.facecolor": "white",
    "xtick.color":      "black",
    "ytick.color":      "black",
    "xtick.direction":  "out",
    "ytick.direction":  "out",
    "xtick.major.width":1,
    "ytick.major.width":1,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
})

MM = 1 / 25.4
FIG_COL1_IN, FIG_COL2_IN = 84 * MM, 174 * MM

def save_figure(fig, path_stem, formats=("png","svg")):
    """Save the figure in each of `formats` at its own size"""
    w, h = fig.get_size_inches()
    for fmt in formats:
        fig.savefig(Path(path_stem).with_suffix(f".{fmt}"), format=fmt, dpi=1200)
    print(f"[save_figure] {w / MM:.0f} x {h / MM:.0f} mm -> {Path(path_stem).name}")

In [2]:
DEFAULT_COLUMN_ORDER = [
    "Subject.Name", "Subject.People", "Subject.Species", "Subject.Sex", "Subject.Box", "Subject.Order",
    "Subject.DOB", "Subject.Success", "Subject.Include", "Subject.Group",
    "Block Number", "Block", "Block ID", "Block Day", "Block Retry", "Block Renumber", "Date",
    "Overall Day", "Session", "File", "File Count", "Criteria Met", "Index",
    "Trial Num", "Trial Type Num", "Time", "Hour", "Stimulus", "Trial Type",
    "Class", "Tempo", "Tempo Bin", "Bin Number", "d Prime", "Beta", "Criterion", "Binomial p", "Hit Rate", "FA Rate",
    "CR Rate", "Overall Acc", "Hit Rate - FA Rate", "RT mean", "RT std", "RT median", "RT",
    "Timeout", "Response", "Response Type", "Reward", "Punish", "Total Hit",
    "Total Miss", "Total FA", "Total CR", "NR", "Hit", "Miss", "Miss (NR)",
    "FA", "CR", "CR (NR)", "S+ Trials", "S+ (NR) Trials", "S- Trials",
    "S- (NR) Trials", "Trials", "Probe Hit", "Probe Miss", "Probe Miss (NR)",
    "Probe FA", "Probe CR", "Probe CR (NR)", "Probe S+ Trials",
    "Probe S+ (NR) Trials", "Probe S- Trials", "Probe S- (NR) Trials",
    "Probe Trials",
]

COUNT_COLUMNS = [
    "CR", "CR (NR)", "FA", "Hit", "Miss", "Miss (NR)", "Probe CR",
    "Probe CR (NR)", "Probe FA", "Probe Hit", "Probe Miss", "Probe Miss (NR)",
    "Probe S+ (NR) Trials", "Probe S+ Trials", "Probe S- (NR) Trials",
    "Probe S- Trials", "Probe Trials", "Reward", "S+ (NR) Trials", "S+ Trials",
    "S- (NR) Trials", "S- Trials", "Session", "Trials", "Hour", "Block Number",
    "File Count", "Index",
]

FLOAT_COLUMNS = ["RT", "Timeout", "Tempo"]

SYLLABLE_DUR_MS = {"BF": 38, "ZF": 75}  # syllable length subtracted from IOI to get gap duration

# Mapping from sound to testing block names
PROBE_ABC_BLOCKS = {
    "Sound A": ["testing 144 (120/180)", "testing 144 (120/180) bengalese"],
    "Sound B": ["testing 144b (120/180)", "testing 144b (120/180) bengalese"],
    "Sound C": ["testing 144c (120/180)", "testing 144c (120/180) bengalese"],
}

PROBE_ABC_VALID_BLOCKS = [block for blocks in PROBE_ABC_BLOCKS.values() for block in blocks]
PROBE_SOUND_MAP = {block: sound for sound, blocks in PROBE_ABC_BLOCKS.items() for block in blocks}

# Mapping from sound to training block names
TRAINING_ABC_BLOCKS = {
        "Sound A": ["training 120/180 bengalese", "training 120/180"],
        "Sound B": ["training 120/180b bengalese", "training 120/180b"],
        "Sound C": ["training 120/180c bengalese", "training 120/180c"],
}

TRAINING_ONLY_SOUND_MAP = {block: sound for sound, blocks in TRAINING_ABC_BLOCKS.items() for block in blocks}

# Criteria
CRITERION_HIT_RATE = 0.60
CRITERION_CR_RATE = 0.60
CRITERION_OVERALL_RATE = 0.75
CRITERION_WINDOW_DAYS = 3
CRITERION_REQUIRED_DAYS = 2
CRITERION_MAX_BLOCK_DAY = 30

# Manual corrections to fix data errors
MANUAL_BLOCK_NUMBER_CORRECTIONS = (
    dict(subject_contains='o78w78', block_contains='training 120/180',
         block_number=6, new_block_number=4,
         reason='o78w78: merge training around an accidental early reduced-reinforcement block'),
)

MANUAL_ROW_EXCLUSIONS = (
    dict(block='40-240d bengalese', tempo_gt=240, reason='IOI outside the BF rule-training range'),
    dict(subject='g14o81', timeout_gt=30, reason='g14o81: timeout typo'),
    dict(subject='g14o81', blocks=('testing 144 (120/180) bengalese',
                                    'testing 144b (120/180) bengalese'),
         reason='g14o81: probe tests after below-criterion reduced reinforcement'),
    dict(subject='g14o81', block='testing 144c (120/180) bengalese', dates=('2025-08-05',),
         reason='g14o81: <100-trial Sound C testing day'),
    dict(subject='w66w67', block='testing 144b (120/180) bengalese',
         dates=('2026-01-23', '2026-01-24'), reason='w66w67: <100-trial Sound B testing days'),
    dict(subject='y4p74', block='testing 144b (120/180) bengalese', dates=('2026-06-20',),
         reason='y4p74: <100-trial Sound B testing day'),
    dict(subjects=('pi77pi8', 'b0r34'),
         block_not_in=('discrim 120:144 S+/S- bengalese', 'discrim 144:120 S+/S- bengalese',
                       'discrim 120:132 S+/S- bengalese', 'discrim 132:120 S+/S- bengalese',
                       'discrim pitch e hi/lo S+/S- bengalese',
                       'discrim pitch e lo/hi S+/S- bengalese',
                       'discrim amp e hi/lo S+/S- bengalese',
                       'discrim amp e lo/hi S+/S- bengalese'),
         reason='pi77pi8 / b0r34: retain only discrimination blocks for discrimination birds'),
    dict(subject='o62p45', block='training 120/180 bengalese',
         reason='o62p45: training trials recorded for a discrimination bird'),
    dict(subject='r100p13', block='training 120/180b',
         reason='r100p13: accidentally did 1 trial of sound b'),
)

@dataclass(frozen=True)
class AnalysisPaths:
    """Filesystem locations used by the analysis notebook."""
    raw_data: Path
    output: Path
    subject_file: Path
    cache_file: Path = LOCAL_DIR / "AllData.csv"

In [3]:
class SignalDetectionStats:
    """Vectorized signal-detection and accuracy calculations."""
    def __init__(self, n_hit: np.ndarray, n_miss: np.ndarray, n_fa: np.ndarray, n_cr: np.ndarray):
        self.n_hit = n_hit.astype(float)
        self.n_miss = n_miss.astype(float)
        self.n_fa = n_fa.astype(float)
        self.n_cr = n_cr.astype(float)

        self.n_splus = self.n_hit + self.n_miss
        self.n_sminus = self.n_fa + self.n_cr

        # Rows with no S+ or no S- trials (n < 1) temporarily hold -1 instead:
        # that makes the half-count nudge in dprime_beta_criterion negative and
        # pushes the corrected rate outside (0, 1), so the row can be NaN'd
        # (d' is undefined with no trials of a stimulus class).
        self.n_splus_safe = np.where(self.n_splus < 1, -1, self.n_splus)
        self.n_sminus_safe = np.where(self.n_sminus < 1, -1, self.n_sminus)

        self.n_correct = self.n_hit + self.n_cr
        self.n_incorrect = self.n_miss + self.n_fa

    @classmethod
    def from_frame(
        cls,
        frame: pd.DataFrame,
        hit_col: str = "Total Hit",
        miss_col: str = "Total Miss",
        fa_col: str = "Total FA",
        cr_col: str = "Total CR",
    ) -> "SignalDetectionStats":
        """Build from a frame's count columns."""
        return cls(
            frame[hit_col].to_numpy(dtype=float),
            frame[miss_col].to_numpy(dtype=float),
            frame[fa_col].to_numpy(dtype=float),
            frame[cr_col].to_numpy(dtype=float),
        )

    def dprime_beta_criterion(self) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Calculate d', beta, and criterion.

        d' = z(hit rate) - z(FA rate)
        beta = exp((z(FA rate)^2 - z(hit rate)^2) / 2)
        criterion = -0.5 * (z(hit rate) + z(FA rate))

        Rates of exactly 0 or 1 give infinite z-scores, so they are nudged
        half a trial inward (the 1/(2n) log-linear correction), following
        Macmillan & Kaplan (1985). The process is:
         Rows with no S+ or no S- trials carry -1 in n_splus_safe /
         n_sminus_safe (set in __init__), so the nudge is negative and the
         corrected rate falls outside (0, 1) instead of raising a
         divide-by-zero error.
         Those rows are then identified by the `impossible` mask and get
         NaN, since d' is undefined without trials of both classes.
        """
        hit_rate = self.n_hit / self.n_splus_safe
        fa_rate = self.n_fa / self.n_sminus_safe

        nudge_hit = 1.0 / (2.0 * self.n_splus_safe)
        nudge_fa = 1.0 / (2.0 * self.n_sminus_safe)

        # Correction if hit or FA rate equals 0 or 1 (Macmillan & Kaplan, 1985)
        hit_rate = np.where(hit_rate >= 1, 1 - nudge_hit, hit_rate)
        hit_rate = np.where(hit_rate <= 0, 0 + nudge_hit, hit_rate)
        fa_rate = np.where(fa_rate >= 1, 1 - nudge_fa, fa_rate)
        fa_rate = np.where(fa_rate <= 0, 0 + nudge_fa, fa_rate)

        z_hit = stats.norm.ppf(hit_rate)
        z_fa = stats.norm.ppf(fa_rate)

        d_prime = z_hit - z_fa
        beta = np.exp((z_fa ** 2 - z_hit ** 2) / 2)
        criterion = -0.5 * (z_hit + z_fa)

        # Rows with no S+ or no S- trials: the negative nudge left their
        # corrected rate outside (0, 1) -> d', beta, and criterion are NaN.
        impossible = (
            (hit_rate > 1) | (hit_rate < 0) |
            (fa_rate > 1) | (fa_rate < 0)
        )

        d_prime = np.where(impossible, np.nan, d_prime)
        beta = np.where(impossible, np.nan, beta)
        criterion = np.where(impossible, np.nan, criterion)

        return d_prime, beta, criterion

    def basic_rates(self) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """Calculate Hit / CR / FA / overall / hit-minus-FA rates."""
        with np.errstate(divide="ignore", invalid="ignore"):
            hit_rate = self.n_hit / self.n_splus
            cr_rate = self.n_cr / self.n_sminus
            fa_rate = self.n_fa / self.n_sminus
            overall = self.n_correct / (self.n_correct + self.n_incorrect)
        hit_minus_fa = hit_rate - fa_rate
        return hit_rate, cr_rate, fa_rate, overall, hit_minus_fa

    def binomial_p(self) -> np.ndarray:
        """One-tailed binomial p"""
        total = self.n_correct + self.n_incorrect
        out = np.full_like(total, fill_value=np.nan, dtype=float)
        for i, (k, n) in enumerate(zip(self.n_correct, total)):
            if n <= 0:
                continue
            out[i] = stats.binomtest(int(k), int(n), p=0.5, alternative="greater").pvalue
        return out

    def criteria_met(self) -> np.ndarray:
        """Rows meeting all criterion thresholds"""
        with np.errstate(divide="ignore", invalid="ignore"):
            hit_rate = self.n_hit / self.n_splus_safe
            cr_rate = self.n_cr / self.n_sminus_safe
            overall = (self.n_hit + self.n_cr) / (self.n_hit + self.n_miss + self.n_fa + self.n_cr)
        return (overall >= CRITERION_OVERALL_RATE) & (hit_rate >= CRITERION_HIT_RATE) & (cr_rate >= CRITERION_CR_RATE)

In [4]:
def _read_wav_rms(wav_path: Path) -> float:
    """RMS amplitude of a wav file"""
    try:
        _, data = wavfile.read(wav_path)
        arr = data.astype(np.float64)
        if arr.ndim > 1:
            arr = arr.mean(axis=1)
        if np.issubdtype(data.dtype, np.integer):
            arr /= np.iinfo(data.dtype).max
        return float(np.sqrt(np.mean(arr ** 2)))
    except Exception as e:
        print(f"Warning: could not compute RMS for {wav_path.name}: {e}")
        return np.nan


def _parse_stimulus_header(header_path: Path):
    """Parse a stimulus header file -> (avg_ioi, target_ioi, stddev, start_times); missing fields are None."""
    avg_ioi = target_ioi = interval_stddev = None
    start_times = []
    in_table = False

    with header_path.open("r") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line:
                continue
            if "Actual Avg IOI:" in line:
                avg_ioi = float(line.split("Actual Avg IOI:")[1].split()[0])
            if "Target IOI:" in line:
                target_ioi = float(line.split("Target IOI:")[1].split()[0])
            if "Interval StdDev:" in line:
                interval_stddev = float(line.split("Interval StdDev:")[1].split()[0])
            elif "Interval (samp)" in line and "Start Time (s)" in line:
                in_table = True
            elif in_table:
                parts = line.split()
                if len(parts) >= 3:
                    try:
                        float(parts[1])  # validate the interval column
                        start_times.append(float(parts[2]))
                    except ValueError:
                        pass

    return avg_ioi, target_ioi, interval_stddev, start_times


class BirdAnalysis:
    """Analysis wrapper for bird data."""

    def __init__(self, paths, reimport="auto", verbose=False, rt_offset_seconds=0.5):
        """Load subject data."""
        self.paths = paths
        self.verbose = verbose
        self.paths.output.mkdir(parents=True, exist_ok=True)
        self.rt_offset_seconds = rt_offset_seconds
        self._criterion_table_cache = None

        if self.verbose:
            print("[BirdAnalysis] Creating analysis object...")
            print("[BirdAnalysis] Loading subject metadata...")
        self._load_subject_info()

        if self.verbose:
            print(f"[BirdAnalysis] Loading behavioral data (reimport={reimport})...")
        self._load_or_build_all_data(reimport=reimport)

        if self.verbose:
            print("[BirdAnalysis] All Complete.")

    # ---------------------------- loading ---------------------------- #
    def _load_subject_info(self):
        """Load and filter the subject metadata sheet."""
        bird_stats = pd.read_excel(self.paths.subject_file).copy()

        if "Bird" in bird_stats.columns and "Name" not in bird_stats.columns:
            bird_stats = bird_stats.rename(columns={"Bird": "Name"})

        bird_stats = bird_stats.add_prefix("Subject.")

        for col in ["Subject.DOB", "Subject.Arrived"]:
            if col in bird_stats.columns:
                bird_stats[col] = pd.to_datetime(bird_stats[col], errors="coerce").dt.date

        if "Subject.Include" in bird_stats.columns:
            bird_stats["Subject.Include"] = bird_stats["Subject.Include"].fillna(False).astype(bool)

        if "Subject.Arrived" in bird_stats.columns:
            bird_stats = bird_stats.loc[bird_stats["Subject.Arrived"].notna()].copy()

        if "Subject.Include" in bird_stats.columns:
            bird_stats = bird_stats.loc[bird_stats["Subject.Include"]].copy()

        self.bird_stats = bird_stats.reset_index(drop=True)
        self.bird_list = (
            self.bird_stats["Subject.Name"]
            .dropna()
            .astype(str)
            .sort_values()
            .unique()
            .tolist()
        )

    def _should_reimport(self, reimport):
        """Rebuild all_data?"""
        cache_exists = self.paths.cache_file.exists()

        if isinstance(reimport, bool):
            return reimport or not cache_exists

        if reimport is None or str(reimport).lower() == "auto":
            if not cache_exists:
                return True
            modified = dt.datetime.fromtimestamp(self.paths.cache_file.stat().st_mtime)
            hours_since_modified = (dt.datetime.now() - modified).total_seconds() / 3600
            return hours_since_modified > 12

        raise ValueError("reimport must be True, False, 'auto', or None.")

    def _load_or_build_all_data(self, reimport):
        """Load all_data from cache, or rebuild from raw CSVs."""
        cache_path = self.paths.cache_file
        cache_path.parent.mkdir(parents=True, exist_ok=True)

        if self._should_reimport(reimport):
            if self.verbose:
                print("[BirdAnalysis] Rebuilding all_data from raw csv files...")
            self.all_data = self._build_all_data_from_raw()
            self._add_block_sequence_columns()
            self.all_data.to_csv(cache_path, index=False)
            if self.verbose:
                print(f"[BirdAnalysis] Saved cache to {cache_path}")
            return

        if self.verbose:
            print(f"[BirdAnalysis] Loading cached all_data from {cache_path}")
        self.all_data = pd.read_csv(cache_path)
        self._coerce_columns(self.all_data)
        self._add_block_sequence_columns()
        self.all_data = self.reorder_columns(self.all_data)

    def _build_all_data_from_raw(self):
        """Concatenate raw per-bird CSVs; apply manual fixes + derived columns."""
        frames = []

        for csv_path in sorted(self.paths.raw_data.glob("*.csv")):
            bird_name = csv_path.stem
            if bird_name not in self.bird_list:
                continue
            if self.verbose:
                print(f"building {bird_name}...")
            curr = pd.read_csv(csv_path, low_memory=False)
            self._coerce_columns(curr)
            frames.append(curr)

        if not frames:
            raise FileNotFoundError(f"No subject csv files were found in {self.paths.raw_data}")

        all_data = pd.concat(frames, ignore_index=True)
        all_data = self._apply_manual_fixes(all_data)
        all_data = self._add_derived_columns(all_data)
        return self.reorder_columns(all_data)

    def _coerce_columns(self, frame):
        """Cast raw columns to proper dtypes (count columns fill NaN with 0)."""
        for col in COUNT_COLUMNS:
            if col in frame.columns:
                frame[col] = pd.to_numeric(frame[col], errors="coerce").fillna(0)
        for col in FLOAT_COLUMNS:
            if col in frame.columns:
                frame[col] = pd.to_numeric(frame[col], errors="coerce")
        if "Date" in frame.columns:
            frame["Date"] = pd.to_datetime(frame["Date"], errors="coerce", format="mixed")
        # Time stays a string: mixed formats sort correctly.

    @staticmethod
    def _manual_rule_mask(frame, rule):
        """Row mask for one correction rule; runs on raw rows (subjects are in 'Subject')."""
        valid_fields = {
            'subject', 'subjects', 'subject_contains', 'block', 'blocks',
            'block_not_in', 'block_contains', 'block_number', 'block_numbers',
            'date_before', 'dates', 'tempo_gt', 'timeout_gt',
            'new_block_number', 'reason',
        }
        unknown_fields = set(rule) - valid_fields
        if unknown_fields:
            raise ValueError(f'Unknown manual-correction fields: {sorted(unknown_fields)}')

        mask = pd.Series(True, index=frame.index)
        if 'subject' in rule:
            mask &= frame['Subject'].eq(rule['subject'])
        if 'subjects' in rule:
            mask &= frame['Subject'].isin(rule['subjects'])
        if 'subject_contains' in rule:
            mask &= frame['Subject'].str.contains(rule['subject_contains'], na=False, regex=False)
        if 'block' in rule:
            mask &= frame['Block'].eq(rule['block'])
        if 'blocks' in rule:
            mask &= frame['Block'].isin(rule['blocks'])
        if 'block_not_in' in rule:
            mask &= ~frame['Block'].isin(rule['block_not_in'])
        if 'block_contains' in rule:
            mask &= frame['Block'].str.contains(rule['block_contains'], na=False, regex=False)
        if 'block_number' in rule:
            mask &= frame['Block Number'].eq(rule['block_number'])
        if 'block_numbers' in rule:
            mask &= frame['Block Number'].isin(rule['block_numbers'])
        if 'date_before' in rule:
            mask &= frame['Date'].lt(pd.Timestamp(rule['date_before']))
        if 'dates' in rule:
            mask &= frame['Date'].dt.normalize().isin(pd.to_datetime(rule['dates']))
        if 'tempo_gt' in rule:
            mask &= pd.to_numeric(frame['Tempo'], errors='coerce').gt(rule['tempo_gt'])
        if 'timeout_gt' in rule:
            mask &= pd.to_numeric(frame['Timeout'], errors='coerce').gt(rule['timeout_gt'])
        return mask

    def _apply_manual_fixes(self, frame):
        """Apply block-number corrections and row exclusions."""
        frame = frame.copy()
        print('Applying manual data corrections...')

        for rule in MANUAL_BLOCK_NUMBER_CORRECTIONS:
            mask = self._manual_rule_mask(frame, rule)
            frame.loc[mask, 'Block Number'] = rule['new_block_number']
            print(f"  corrected {int(mask.sum()):,} rows: {rule['reason']}")

        for rule in MANUAL_ROW_EXCLUSIONS:
            mask = self._manual_rule_mask(frame, rule)
            print(f"  excluded {int(mask.sum()):,} rows: {rule['reason']}")
            frame = frame.loc[~mask].copy()

        return frame


    def _add_derived_columns(self, frame):
        """Merge subject metadata and add derived per-trial columns."""
        data = frame.copy()

        if "Subject" in data.columns and "Subject.Name" not in data.columns:
            data = data.merge(self.bird_stats, how="left", left_on="Subject", right_on="Subject.Name")
        elif "Subject.Name" not in data.columns:
            raise KeyError("Expected either 'Subject' or 'Subject.Name' in the raw data.")

        data = data.drop(columns=["Subject"], errors="ignore")

        if "RT" in data.columns:
            data["RT"] = np.where(data["RT"].isna(), np.nan, data["RT"] + self.rt_offset_seconds)

        if {"Block", "Block Number"}.issubset(data.columns):
            data["Block ID"] = data["Block"].astype(str) + " " + data["Block Number"].astype("Int64").astype(str)

        if {"Class", "Trial Type"}.issubset(data.columns):
            probe_mask = data["Class"].isin(["probePlus", "probeMinus"])
            data.loc[probe_mask, "Trial Type"] = "probe"

        if "Tempo" in data.columns:
            data.loc[data["Tempo"].astype(str).str.lower() == "shaping", "Tempo"] = 0
            data["Tempo"] = pd.to_numeric(data["Tempo"], errors="coerce")

        sort_cols = [col for col in ["Subject.Name", "Block ID", "Date", "Time"] if col in data.columns]
        if sort_cols:
            data = data.sort_values(sort_cols).reset_index(drop=True)

        if {"Subject.Name", "Block ID"}.issubset(data.columns):
            data["Trial Num"] = data.groupby(["Subject.Name", "Block ID"]).cumcount() + 1

        if {"Subject.Name", "Block ID", "Trial Type"}.issubset(data.columns):
            data["Trial Type Num"] = data.groupby(["Subject.Name", "Block ID", "Trial Type"]).cumcount() + 1

        count_pairs = {
            # NR means no response.
            "NR": ["Miss (NR)", "CR (NR)"],
            "Total Hit": ["Hit", "Probe Hit"],
            "Total Miss": ["Miss", "Miss (NR)", "Probe Miss", "Probe Miss (NR)"],
            "Total FA": ["FA", "Probe FA"],
            "Total CR": ["CR", "CR (NR)", "Probe CR", "Probe CR (NR)"],
        }
        for new_col, source_cols in count_pairs.items():
            data[new_col] = sum(data.get(col, 0) for col in source_cols)

        if {"Subject.Name", "Block ID", "Date"}.issubset(data.columns):
            day_min = data.groupby(["Subject.Name", "Block ID"])["Date"].transform("min")
            data["Block Day"] = (data["Date"] - day_min).dt.days

        if {"Subject.Name", "Date"}.issubset(data.columns):
            overall_min = data.groupby("Subject.Name")["Date"].transform("min")
            data["Overall Day"] = (data["Date"] - overall_min).dt.days

        return data

    # ---------------------------- utilities ---------------------------- #
    @staticmethod
    def reorder_columns(frame, column_order=None):
        """Reorder to DEFAULT_COLUMN_ORDER; unknown columns."""
        column_order = DEFAULT_COLUMN_ORDER if column_order is None else column_order
        existing = list(frame.columns)
        ordered = [col for col in column_order if col in existing]
        trailing = [col for col in existing if col not in ordered]
        return frame.loc[:, ordered + trailing]

    def _add_block_sequence_columns(self):
        """Add block identity/order columns once when all_data is loaded."""
        self.all_data['Block Number'] = pd.to_numeric(
            self.all_data['Block Number'], errors='coerce')
        # Block Retry keeps track of number of time a bird has done the same block. Block Retry == 1 means only
        # get the first attempt of blocks
        self.all_data['Block Retry'] = (
            self.all_data.groupby(['Subject.Name', 'Block'])['Block Number']
                         .rank(method='dense').astype('Int64')
        )
        self.all_data['Block Renumber'] = (
            self.all_data.groupby('Subject.Name')['Block Number']
                         .rank(method='dense').astype('Int64')
        )

    @staticmethod
    def _bin_trials(frame, groups, column=None, new_column="Bin", bin_size=40, relabel=True, add_bin_numbers=False, bin_start=None):
        """Bin trials into `bin_size`-wide bins; returns the binned frame."""
        data = frame.copy()
        group_cols = list(groups)

        if column is None:
            data[new_column] = data.groupby(group_cols).cumcount() + 1
            column = new_column

        non_null = data[column].dropna()
        if non_null.empty:
            return data.iloc[0:0]

        if bin_start is None:
            bin_start = non_null.min()

        bins = pd.interval_range(start=bin_start, end=non_null.max() + bin_size, freq=bin_size, closed="left")
        data[new_column] = pd.cut(data[column], bins=bins)

        if relabel:
            data[new_column] = data[new_column].cat.codes
        elif add_bin_numbers:
            data["Bin Number"] = data[new_column].cat.codes

        return data.dropna(subset=[new_column])

    def _summarize_grouped(self, grouped):
        """Aggregate a GroupBy into count / RT summaries."""
        groupings = {
            "Total Hit": pd.NamedAgg(column="Total Hit", aggfunc="sum"),
            "Total Miss": pd.NamedAgg(column="Total Miss", aggfunc="sum"),
            "Total FA": pd.NamedAgg(column="Total FA", aggfunc="sum"),
            "Total CR": pd.NamedAgg(column="Total CR", aggfunc="sum"),
            "Probe Total S+": pd.NamedAgg(column="Probe S+ (NR) Trials", aggfunc="sum"),
            "Probe Total S-": pd.NamedAgg(column="Probe S- (NR) Trials", aggfunc="sum"),
            "RT mean": pd.NamedAgg(column="RT", aggfunc="mean"),
            "RT median": pd.NamedAgg(column="RT", aggfunc="median"),
            "RT std": pd.NamedAgg(column="RT", aggfunc="std"),
            "Timeout": pd.NamedAgg(column="Timeout", aggfunc="mean"),
            "Trials": pd.NamedAgg(column="Trials", aggfunc="sum"),
            "Probe Trials": pd.NamedAgg(column="Probe Trials", aggfunc="sum"),
        }

        names = list(grouped.keys)
        if "Block Day" in names and "Overall Day" not in names:
            groupings["Overall Day"] = pd.NamedAgg(column="Overall Day", aggfunc="min")
        if "Overall Day" in names and "Block Day" not in names:
            groupings["Block Day"] = pd.NamedAgg(column="Block Day", aggfunc="min")
        if 'Block ID' in names:
            for column in ['Block', 'Block Number', 'Block Retry', 'Block Renumber']:
                if column not in names:
                    groupings[column] = pd.NamedAgg(column=column, aggfunc='first')

        grouped_data = grouped.agg(**groupings).dropna(how="all")

        if not grouped_data.empty and "Subject.Name" in grouped_data.index.names:
            grouped_data = grouped_data.join(self.bird_stats.set_index("Subject.Name"), on="Subject.Name")

        return grouped_data

    def _apply_group_stats(self, frame):
        """Attach SDT metrics to a summarized frame."""
        if frame.empty:
            return frame.copy()

        out = frame.copy()
        stats_obj = SignalDetectionStats.from_frame(out)

        d_prime, beta, criterion = stats_obj.dprime_beta_criterion()
        hit_rate, cr_rate, fa_rate, overall, hit_minus_fa = stats_obj.basic_rates()

        out["d Prime"] = d_prime
        out["Beta"] = beta
        out["Criterion"] = criterion
        out["Binomial p"] = stats_obj.binomial_p()
        out["Hit Rate"] = hit_rate
        out["FA Rate"] = fa_rate
        out["CR Rate"] = cr_rate
        out["Overall Acc"] = overall
        out["Hit Rate - FA Rate"] = hit_minus_fa

        return self.reorder_columns(out)

    def run_analysis(self, frame, groups, bin_bool=False, new_column="Bin", column=None, bin_size=40, relabel=True, add_bin_numbers=False, bin_start=None):
        """Group + summarize + score a frame (optionally binned)."""
        if frame.empty:
            return pd.DataFrame()

        if bin_bool:
            frame = self._bin_trials(
                frame, groups=groups, column=column, new_column=new_column,
                bin_size=bin_size, relabel=relabel, add_bin_numbers=add_bin_numbers,
                bin_start=bin_start,
            )
            groups = list(groups) + ([new_column] if new_column in frame.columns else [])

        summarized = self._summarize_grouped(frame.groupby(list(groups), observed=True))
        summarized = self._apply_group_stats(summarized)
        return self.reorder_columns(summarized)

    def extract_stimulus_info(self, species, vrms=False, stim_path=STIM_DIR / 'Rule_Training_40-275' / 'Stimulus Identification.txt', core_dir=STIM_DIR / 'Rule_Training_40-275', prefix_zf='z'):
        """Stimulus stats (CV, V_RMS, gap) for one species."""
        header_dir = core_dir / "File Data"
        wav_dir    = core_dir / "Stim Files"

        ids = {}
        with stim_path.open("r") as handle:
            for line in handle:
                line = line.strip()
                if not line or "-" not in line: continue
                if species == "BF" and "BF" not in line: continue
                if species != "BF" and "ZF" not in line: continue
                value, key = line.split("-", 1)
                ids[key.strip()] = value.strip()

        if species == "BF":
            syllable_duration = SYLLABLE_DUR_MS["BF"] / 1000
            stim_names = sorted(stim.name for stim in header_dir.glob("*.txt") if stim.name.startswith("g"))
        else:
            syllable_duration = SYLLABLE_DUR_MS["ZF"] / 1000
            stim_names = sorted(stim.name for stim in header_dir.glob("*.txt") if stim.name.startswith(prefix_zf))

        records = []
        for key, block in ids.items():
            for stim_name in sorted(name for name in stim_names if key in name):
                wav_path = wav_dir / stim_name.replace(".txt", ".wav")
                v_rms = _read_wav_rms(wav_path) if (vrms and wav_path.exists()) else np.nan

                avg_ioi, target_ioi, interval_stddev, start_times = _parse_stimulus_header(header_dir / stim_name)
                if avg_ioi is None or target_ioi is None or interval_stddev is None:
                    continue

                record = {
                    "Stimulus": stim_name,
                    "Block": block,
                    "Actual Avg IOI (ms)": avg_ioi * 1000,
                    "CV": interval_stddev / avg_ioi,
                    "Gap Duration ms": np.round((target_ioi - syllable_duration) * 1000),
                    "number_of_sounds": len(start_times),
                }
                if vrms:
                    record["V_RMS"] = v_rms
                records.append(record)

        out = pd.DataFrame.from_records(records)
        return out.sort_values(["Block", "Actual Avg IOI (ms)", "Stimulus"]).reset_index(drop=True)

    def _extract_training_probe_info(self, base_dir, species=None):
        """Training/probe stimulus stats; same parsing as extract_stimulus_info, plus a Sound column."""
        base_dir = Path(base_dir)
        header_dir = base_dir / "File Data"
        wav_dir = base_dir / "Stim Files"
        species = species.upper() if species is not None else None

        # Sound per file from this folder's Stimulus Identification.txt (prefix -> sound).
        sound_id_map = {}
        stim_id_path = base_dir / "Stimulus Identification.txt"
        if stim_id_path.exists():
            with stim_id_path.open("r") as id_handle:
                for id_line in id_handle:
                    id_line = id_line.strip()
                    if not id_line or "-" not in id_line:
                        continue
                    if species == "BF" and "BF" not in id_line:
                        continue
                    if species == "ZF" and "ZF" not in id_line:
                        continue
                    label, key = id_line.split("-", 1)
                    label, key = label.strip(), key.strip()
                    if "Rule" in label:
                        continue
                    sound_id_map[key] = label.replace("BF ", "").replace("ZF ", "")

        stim_files = sorted(header_dir.glob("*.txt"))
        if species is not None:
            prefix = "g" if species == "BF" else "z"
            stim_files = [f for f in stim_files if f.name.startswith(prefix)]
        else:
            stim_files = [f for f in stim_files if f.name.startswith(("g", "z"))]

        records = []
        for stim_file in stim_files:
            stim_name = stim_file.name
            stim_species = "BF" if stim_name.startswith("g") else "ZF"
            syllable_duration = SYLLABLE_DUR_MS[stim_species] / 1000

            sound = next((lbl for key, lbl in sound_id_map.items() if key in stim_file.stem), None)
            if sound is None:
                continue

            wav_path = wav_dir / stim_file.with_suffix(".wav").name
            v_rms = _read_wav_rms(wav_path) if wav_path.exists() else np.nan

            avg_ioi, target_ioi, interval_stddev, start_times = _parse_stimulus_header(stim_file)
            if avg_ioi is None or interval_stddev is None:
                continue

            records.append({
                "Stimulus": stim_name,
                "Species": stim_species,
                "Block": "Training Probe",
                "Actual Avg IOI (ms)": avg_ioi * 1000,
                "CV": interval_stddev / avg_ioi,
                "V_RMS": v_rms,
                "Gap Duration ms": (np.round((target_ioi - syllable_duration) * 1000)
                                    if target_ioi is not None else np.nan),
                "number_of_sounds": len(start_times),
                "Sound": sound,
            })

        return (pd.DataFrame.from_records(records)
                .sort_values(["Sound", "Actual Avg IOI (ms)", "Stimulus"])
                .reset_index(drop=True))

    # ---------------------------- computation ---------------------------- #

    def criteria_from_counts(self, frame):
        """Criterion-passed mask for a summarized frame."""
        return SignalDetectionStats.from_frame(frame).criteria_met()

    def compute_criterion_table(self, refresh=False):
        """One daily criterion table shared by TTC and learning curves."""
        if self._criterion_table_cache is not None and not refresh:
            return self._criterion_table_cache.copy()

        keys = ['Subject.Name', 'Block ID']
        daily = self.run_analysis(self.all_data, keys + ['Block Day']).reset_index()
        daily = daily[(daily['Block Retry'] == 1) &
                      (daily['Block Day'] <= CRITERION_MAX_BLOCK_DAY)].copy()
        daily = daily.sort_values(keys + ['Block Day'])
        daily['Criteria Met'] = self.criteria_from_counts(daily)
        daily['Criterion Window'] = (
            daily.groupby(keys)['Criteria Met']
                 .transform(lambda passed: passed.rolling(
                     CRITERION_WINDOW_DAYS, min_periods=1).sum())
        )

        criterion_days = (
            daily.loc[daily['Criterion Window'] >= CRITERION_REQUIRED_DAYS]
                 .groupby(keys)['Block Day'].min()
                 .rename('Criterion Day')
        )

        daily = daily.set_index(keys).join(criterion_days).reset_index()
        self._criterion_table_cache = daily
        return daily.copy()

    def compute_trials_to_criteria(self):
        """Trials to criterion per bird x block."""
        group_list = ["Subject.Name", "Block ID"]

        block_order = self.all_data.groupby(group_list).agg(
            **{
                "Subject.Species": ("Subject.Species", "first"),
                "Block": ("Block", "first"),
                "Block_Number": ("Block Number", "first"),
                "Block Retry": ("Block Retry", "first"),
                "Block Renumber": ("Block Renumber", "first"),
                "Start_Date": ("Date", "min"),
                "Total_Trials": ("Trials", "sum"),
                "Total_Probe_Trials": ("Probe Trials", "sum"),
            }
        )

        criteria_data = self.compute_criterion_table()
        criterion_days = criteria_data.groupby(group_list)['Criterion Day'].first()
        block_order['Criterion Day'] = criterion_days
        block_order['Block Completed'] = block_order['Criterion Day'].notna()

        completed_days = criteria_data[
            criteria_data['Criterion Day'].notna() &
            (criteria_data['Block Day'] <= criteria_data['Criterion Day'])
        ]
        block_order['Trials to Criteria'] = completed_days.groupby(group_list)['Trials'].sum()
        block_order['Total Hit'] = criteria_data.groupby(group_list)['Total Hit'].last()

        blocks_lower = block_order["Block"].astype(str).str.lower()
        block_order["Block Type"] = np.select(
            [
                blocks_lower.str.contains("shaping"),
                blocks_lower.str.contains("training"),
                blocks_lower.str.contains("testing"),
                blocks_lower.str.contains("range"),
            ],
            ["Shape", "Train", "Test", "Range"],
            default="Other",
        )

        # Completion rules differ by block type (acquisition criteria):
        # Test blocks: at least 80 probe trials completed.
        block_order.loc[block_order["Block Type"] == "Test", "Block Completed"] = block_order["Total_Probe_Trials"] >= 80
        # Range blocks: at least 5000 range trials completed.
        block_order.loc[block_order["Block Type"] == "Range", "Block Completed"] = block_order["Total_Trials"] >= 5000
        # Shaping 1: at least 100 trials completed.
        block_order.loc[blocks_lower.str.contains("shaping 1"), "Block Completed"] = block_order["Total_Trials"] >= 100
        # Shaping 2: at least 10 hits.
        block_order.loc[blocks_lower.str.contains("shaping 2"), "Block Completed"] = block_order["Total Hit"] >= 10

        return block_order.drop(columns=["Total Hit", "Block_Number"], errors="ignore")

    # ------------------------------ plotting ------------------------------ #

    def _first_tempo_discrim_block(self, subj):
        """Earliest 120-vs-144 ms tempo-discrimination block for `subj` (by start date)."""
        d = self.all_data[self.all_data['Subject.Name'] == subj].copy()
        d = d[d['Block'].astype(str).str.contains('120:144|144:120', regex=True, na=False)]
        d = d[d['Block Retry'] == 1]
        return d.groupby('Block')['Date'].min().idxmin()

    def _score_binned_curve(self, d, binsize):
        """Overall / S+ / S- accuracy in complete bins; no-response S- counts as correct."""
        d = d.reset_index(drop=True)
        d["CR Total"]  = d["CR"] + d["CR (NR)"]
        d["Correct"]   = ((d["Class"] == "sPlus") & (d["Hit"] == 1)) | ((d["Class"] == "sMinus") & (d["CR Total"] == 1))
        d["CorrectS+"] = (d["Class"] == "sPlus") & (d["Hit"] == 1)
        d["CorrectS-"] = (d["Class"] == "sMinus") & (d["CR Total"] == 1)

        d = d.iloc[: len(d) // binsize * binsize]  # complete bins only
        d["bin"] = pd.cut(d.index, bins=range(0, len(d) + binsize, binsize), right=False)
        grouped = d.groupby("bin", observed=True)
        nsp = grouped["Class"].apply(lambda x: (x == "sPlus").sum())
        nsm = grouped["Class"].apply(lambda x: (x == "sMinus").sum())
        return (grouped["Correct"].mean().to_numpy(),
                (grouped["CorrectS+"].sum() / nsp).to_numpy(),
                (grouped["CorrectS-"].sum() / nsm).to_numpy())

    def _style_curve_axes(self, ax, n_bins, binsize, fmt, labelbottom, labelleft):
        """Tick/axis styling shared by the learning-curve figures."""
        ax.set_autoscale_on(False)
        ax.set_xlim(-0.5, n_bins - 0.5)
        ax.set_ylim(0, 1.0)
        ax.xaxis.set_major_locator(mticker.MultipleLocator(5000 / binsize))
        ax.xaxis.set_minor_locator(mticker.AutoMinorLocator(2))
        ax.xaxis.set_major_formatter(fmt)
        ax.yaxis.set_major_locator(mticker.FixedLocator([0.0, 0.5, 1.0]))
        ax.yaxis.set_minor_locator(mticker.MultipleLocator(0.125))
        ax.tick_params(which="major", labelsize=11, length=4)
        ax.tick_params(which="minor", length=2)
        ax.tick_params(labelbottom=labelbottom, labelleft=labelleft)
        for mt, loc in zip(ax.yaxis.get_minor_ticks(), ax.yaxis.get_minorticklocs()):
            if abs(loc * 4 - round(loc * 4)) < 1e-9 and abs(loc * 2 - round(loc * 2)) > 1e-9:
                mt.tick1line.set_markersize(3.2)
                mt.tick2line.set_markersize(3.2)
        for sp in ("top", "right"):
            ax.spines[sp].set_visible(False)

    def plot_discrim_learning_curve(self, species='BF', subjects=None, binsize=100,
                                    width_per_bin=0.02, row_h=0.55,
                                    truncate_at_criterion=True):
        """Fig S1B: first tempo-discrimination block, with fixed physical panel height.

        `row_h` is the exported axes height in inches, shared with plot_figS3_group.
        Groups have the same inter-bird spacing; the canvas grows with bird count.
        """
        species = species.upper()
        df = self.all_data[(self.all_data['Subject.Group'] == 'Tempo Discrim') &
                           (self.all_data['Subject.Species'] == species)].copy()
        df = df[df['Block'].astype(str).str.contains('120:144|144:120', regex=True, na=False)]
        df = df[df['Block Retry'] == 1].copy()
        discrim = df['Subject.Name'].unique()
        subjects = sorted(discrim) if subjects is None else [s for s in subjects if s in discrim]

        # Birds that reached criterion on their first attempt.
        first_blocks = {subj: self._first_tempo_discrim_block(subj) for subj in subjects}
        criterion = self.compute_criterion_table()
        criterion_blocks = set(
            criterion.loc[criterion['Criterion Day'].notna(), ['Subject.Name', 'Block']]
                     .itertuples(index=False, name=None)
        )
        subjects = [subj for subj in subjects if (subj, first_blocks[subj]) in criterion_blocks]
        criterion_days = (
            criterion.dropna(subset=['Criterion Day'])
                     .drop_duplicates(['Subject.Name', 'Block'])
                     .set_index(['Subject.Name', 'Block'])['Criterion Day']
        )
        STYLES = {'Combined': {'color': 'black', 'linewidth': 1.5, 'linestyle': '-'},
                  'sPlus':    {'color': 'black', 'linewidth': 0.7, 'linestyle': '-'},
                  'sMinus':   {'color': '0.55',  'linewidth': 1.0, 'linestyle': ':'}}
        SOUND_COLOR = '#9DED90'

        def binned(subj):
            block = self._first_tempo_discrim_block(subj)
            d = df[(df['Subject.Name'] == subj) & (df['Block'] == block)].copy()
            d = d.sort_values(['Date', 'Time'])
            if truncate_at_criterion:
                criterion_day = criterion_days.loc[(subj, block)]
                d = d[d['Block Day'] <= criterion_day]
            return self._score_binned_curve(d, binsize), block

        # Bin each bird and tag by S+ tempo.
        rows = []
        for subj in subjects:
            ser, block = binned(subj)
            tag = '144 S+' if '144:120' in block else '120 S+'
            rows.append((subj, ser, tag))
        if not rows:
            raise ValueError(f"No eligible {species} discrimination birds to plot.")
        max_bins = max(1, max(len(ser[0]) for _, ser, _ in rows))

        groups = [(t, [r for r in rows if r[2] == t]) for t in ('144 S+', '120 S+')]
        groups = [(tag, grp) for tag, grp in groups if grp]

        # ── Layout in inches ──
        ROWGAP = 0.32
        LEFT = 0.50; RIGHT = 0.85; TOP = 0.18; BOTTOM = 0.48
        # Fit only horizontally: never rescale the physical panel height.
        col_w = min(max_bins * width_per_bin, FIG_COL2_IN - LEFT - RIGHT)
        fig_w = LEFT + col_w + RIGHT
        fig_h = TOP + len(rows) * row_h + (len(rows) - 1) * ROWGAP + BOTTOM
        fig = plt.figure(figsize=(fig_w, fig_h))
        fmt = FuncFormatter(lambda v, _: f'{v * binsize / 1000:g}')

        y = TOP
        for tag, grp in groups:
            n = len(grp)
            group_h = n * row_h + (n - 1) * ROWGAP
            fig.text((LEFT + col_w + 0.12) / fig_w,
                     1 - (y + group_h / 2) / fig_h,
                     tag, fontsize=11, ha='left', va='center')
            for r, (subj, ser, _) in enumerate(grp):
                ov, hr, cr = ser
                ax = fig.add_axes([LEFT / fig_w, 1 - (y + row_h) / fig_h,
                                   col_w / fig_w, row_h / fig_h])
                ax.set_facecolor(SOUND_COLOR)
                ax.axhline(0.5, color='black', linestyle=(0, (5, 2)), linewidth=0.7)
                xs = range(len(ov))
                ax.plot(xs, ov, **STYLES['Combined'])
                ax.plot(xs, hr, **STYLES['sPlus'])
                ax.plot(xs, cr, **STYLES['sMinus'])
                self._style_curve_axes(ax, max_bins, binsize, fmt,
                                       labelbottom=(grp is groups[-1][1] and r == n - 1),
                                       labelleft=True)
                y += row_h + ROWGAP
        # Preserve row_h in the export, even when the full stack exceeds page height.
        save_figure(fig, self.paths.output / f'FigS2_{species}_discrim_combined')
        plt.show()
        print(f"{species}: {len(rows)} discrim birds in {len(groups)} S+ groups; "
              f"group max {max_bins} bins")

    def plot_figS3_group(self, species='ZF', orders=('ABC', 'ACB', 'CAB'),
                         subjects=None, binsize=100, width_per_bin=0.0085,
                         row_h=0.3, truncate_at_criterion=True):
        """Fig S1A: uniform bird spacing and fixed exported panel height in inches.

        Column widths and x limits use the longest complete-bin curve at each
        training position across the selected orders (not equal-width sounds).
        Shorter birds share that column's scale. By default curves stop at criterion
        and omit the final incomplete bin, as in _score_binned_curve.
        """
        species = species.upper()
        suffix = ' bengalese' if species == 'BF' else ''
        base_blocks = {'A': f'training 120/180{suffix}',
                       'B': f'training 120/180b{suffix}',
                       'C': f'training 120/180c{suffix}'}
        order_sounds = {'ABC': ['A', 'B', 'C'], 'ACB': ['A', 'C', 'B'], 'CAB': ['C', 'A', 'B']}
        SOUND_COLORS = {'A': '#FFDB99', 'B': '#FAB99B', 'C': '#C08A9B'}
        STYLES = {'Combined': {'color': 'black', 'linewidth': 1.5, 'linestyle': '-'},
                  'sPlus':    {'color': 'black', 'linewidth': 0.7, 'linestyle': '-'},
                  'sMinus':   {'color': '0.55',  'linewidth': 1.0, 'linestyle': ':'}}

        # First attempts only (no retraining appended).
        training_blocks = list(base_blocks.values())
        df = self.all_data[(self.all_data['Subject.Species'] == species) &
                           (self.all_data['Block'].isin(training_blocks))].copy()
        df = df[df['Block Retry'] == 1].copy()
        df = df[df['Subject.Success'] == 1].copy()
        criterion_days = (
            self.compute_criterion_table().dropna(subset=['Criterion Day'])
                .drop_duplicates(['Subject.Name', 'Block'])
                .set_index(['Subject.Name', 'Block'])['Criterion Day']
        )

        def binned(subj, block):
            d = df[(df['Subject.Name'] == subj) & (df['Block'] == block)].copy()
            d = d.sort_values(['Date', 'Time'])
            if truncate_at_criterion:
                criterion_day = criterion_days.loc[(subj, block)]
                d = d[d['Block Day'] <= criterion_day]
            return self._score_binned_curve(d, binsize)

        # Only orders represented by the selected species/subjects.
        order_data = df if subjects is None else df[df['Subject.Name'].isin(subjects)]
        represented_orders = set(order_data['Subject.Order'].dropna().unique())
        orders = tuple(order for order in orders if order in represented_orders)

        groups = []
        for training in orders:
            sounds = order_sounds[training]
            blocks = [base_blocks[s] for s in sounds]
            pool = df[(df['Subject.Order'] == training) &
                      (df['Subject.Species'] == species)]['Subject.Name'].unique()
            subs = sorted(pool) if subjects is None else [s for s in subjects if s in pool]
            ser = {s: [binned(s, b) for b in blocks] for s in subs}
            mb = [max(len(ser[s][i][0]) for s in subs) for i in range(3)]
            groups.append(dict(order=training, subs=subs, ser=ser, mb=mb,
                               colors=[SOUND_COLORS[s] for s in sounds]))
        if not groups:
            raise ValueError(f"No eligible {species} training birds to plot.")
        gmb = [max(1, max(g['mb'][i] for g in groups)) for i in range(3)]

        # ── Layout in inches ──
        COLGAP = 0.35; ROWGAP = 0.32
        LEFT = 0.50; RIGHT = 0.12; TOP = 0.18; BOTTOM = 0.48
        # A common horizontal scale preserves the trial-count width ratios.
        # Limit only the width, so BF and ZF retain exactly the same row_h.
        XU = min(width_per_bin,
                 (FIG_COL2_IN - LEFT - RIGHT - 2 * COLGAP) / sum(gmb))
        content_w = sum(gmb) * XU + 2 * COLGAP
        fig_w = LEFT + content_w + RIGHT
        n_rows = sum(len(g['subs']) for g in groups)
        fig_h = TOP + n_rows * row_h + (n_rows - 1) * ROWGAP + BOTTOM

        fig = plt.figure(figsize=(fig_w, fig_h))
        fmt = FuncFormatter(lambda v, _: f'{v * binsize / 1000:g}')

        y = TOP
        for g in groups:
            n = len(g['subs'])
            for r, subj in enumerate(g['subs']):
                last_row = (g is groups[-1] and r == n - 1)
                xpos = LEFT
                for i in range(3):
                    w = gmb[i] * XU
                    ax = fig.add_axes([xpos / fig_w, 1 - (y + row_h) / fig_h,
                                       w / fig_w, row_h / fig_h])
                    ax.set_facecolor(g['colors'][i])
                    ax.axhline(0.5, color='black', linestyle=(0, (5, 2)), linewidth=0.7)
                    ov, hr, cr = g['ser'][subj][i]
                    xs = range(len(ov))
                    ax.plot(xs, ov, **STYLES['Combined'])
                    ax.plot(xs, hr, **STYLES['sPlus'])
                    ax.plot(xs, cr, **STYLES['sMinus'])
                    self._style_curve_axes(ax, gmb[i], binsize, fmt,
                                           labelbottom=last_row, labelleft=(i == 0))
                    xpos += w + COLGAP
                y += row_h + ROWGAP
        # Preserve row_h in the export, even when the full stack exceeds page height.
        save_figure(fig, self.paths.output / f'FigS3_{species}_combined')
        plt.show()
        print(f"{species}: {sum(len(g['subs']) for g in groups)} birds, "
              f"{len(groups)} order groups; column maxima {gmb}")

    def plot_bird_learning_curve(self, subjects, binsize=100, fig_width_in=2.5, panel_h=0.7,
                                 truncate_at_criterion=True):
        """Learning curves for one bird or a list of birds, one row per bird, one panel per training sound (in training order).

        The figure is exactly `fig_width_in` wide (default: one column, 84 mm) so it drops into
        the page at 100 % with 11 pt text; the width per 100-trial bin is derived from that.
        """
        subjects = [subjects] if isinstance(subjects, str) else list(subjects)
        colors = {"A": "#FFDB99", "B": "#FAB99B", "C": "#C08A9B"}
        crit = (self.compute_criterion_table().dropna(subset=["Criterion Day"])
                .drop_duplicates(["Subject.Name", "Block"]).set_index(["Subject.Name", "Block"])["Criterion Day"])

        rows = []
        for subj in subjects:
            bird = self.all_data[self.all_data["Subject.Name"] == subj]
            if bird.empty:
                raise ValueError(f"{subj} not found in all_data")
            species, order = bird["Subject.Species"].iloc[0], bird["Subject.Order"].iloc[0]
            suffix = " bengalese" if species == "BF" else ""
            blocks = {"A": f"training 120/180{suffix}", "B": f"training 120/180b{suffix}", "C": f"training 120/180c{suffix}"}
            curves = []
            for s in order:
                d = bird[(bird["Block"] == blocks[s]) & (bird["Block Retry"] == 1)].sort_values(["Date", "Time"])
                if truncate_at_criterion and (subj, blocks[s]) in crit.index:
                    d = d[d["Block Day"] <= crit.loc[(subj, blocks[s])]]
                curves.append((s, self._score_binned_curve(d, binsize)))
            rows.append((subj, curves))

        # Layout in inches: columns aligned by training position, column width = longest block at that position.
        # The bin width XU is whatever makes the total width equal fig_width_in.
        COLGAP, ROWGAP, LEFT, RIGHT, TOP, BOTTOM = 0.15, 0.45, 0.7, 0.05, 0.35, 0.55
        col_bins = [max(len(curves[i][1][0]) for _, curves in rows) for i in range(3)]
        XU = (fig_width_in - LEFT - RIGHT - 2 * COLGAP) / sum(max(b, 1) for b in col_bins)
        col_w = [max(b, 1) * XU for b in col_bins]
        fig_w = fig_width_in
        fig_h = TOP + len(rows) * panel_h + (len(rows) - 1) * ROWGAP + BOTTOM
        fig = plt.figure(figsize=(fig_w, fig_h))
        fmt = FuncFormatter(lambda v, _: f"{v * binsize / 1000:g}")

        y = TOP
        for r, (subj, curves) in enumerate(rows):
            x = LEFT
            for i, (s, (ov, hr, cr)) in enumerate(curves):
                ax = fig.add_axes([x / fig_w, 1 - (y + panel_h) / fig_h, col_w[i] / fig_w, panel_h / fig_h])
                ax.set_facecolor(colors[s])
                ax.axhline(0.5, color="black", linestyle=(0, (5, 2)), linewidth=0.7)
                xs = range(len(ov))
                ax.plot(xs, ov, color="black", linewidth=1.5)
                ax.plot(xs, hr, color="black", linewidth=0.7)
                ax.plot(xs, cr, color="0.55", linewidth=1.0, linestyle=":")
                self._style_curve_axes(ax, col_bins[i], binsize, fmt,
                                       labelbottom=(r == len(rows) - 1), labelleft=(i == 0))
                if i == 0:
                    ax.set_title(subj, loc="left", fontsize=11, pad=4)
                x += col_w[i] + COLGAP
            y += panel_h + ROWGAP

        fig.text(0.15 / fig_w, (BOTTOM + (fig_h - TOP - BOTTOM) / 2) / fig_h, "Proportion correct",
                 rotation=90, ha="center", va="center")
        fig.text((LEFT + (sum(col_w) + 2 * COLGAP) / 2) / fig_w, 0.12 / fig_h, "Trials (\u00d710\u00b3)",
                 ha="center", va="bottom")
        save_figure(fig, self.paths.output / f"learning_curve_{'_'.join(subjects)}")
        plt.show()

# Initialization

In [5]:
# Paths and files need to be setup like this for the code to run
paths = AnalysisPaths(
    raw_data=LOCAL_DIR / "Raw Data" / "All Birds",
    output=LOCAL_DIR / "Jupyter Output",
    subject_file=LOCAL_DIR / "Bird_Summary.xlsx",
    cache_file=LOCAL_DIR / "AllData.csv",
)

# set reimport to True if AllData.csv does not exist or you modified the 'All Birds' folder
analysis = BirdAnalysis(paths=paths, reimport=False, verbose=True)

# Summary of birds saved to analysis.all_data
BFs = analysis.all_data[analysis.all_data['Subject.Species'] == 'BF']['Subject.Name'].unique()
ZFs = analysis.all_data[analysis.all_data['Subject.Species'] == 'ZF']['Subject.Name'].unique()
num_BFs = len(BFs)
num_ZFs = len(ZFs)
print(f'{num_BFs} BFs, {num_ZFs} ZFs')
print(f'BFs: {BFs}')
print(f'ZFs: {ZFs}')

[BirdAnalysis] Creating analysis object...
[BirdAnalysis] Loading subject metadata...
[BirdAnalysis] Loading behavioral data (reimport=False)...
[BirdAnalysis] Loading cached all_data from /Users/xutianle/Desktop/Kao Lab/Data Local/AllData.csv


/var/folders/c5/_85xqcr921j_mqh696p2_qnr0000gn/T/ipykernel_31401/2509702213.py:136: DtypeWarning: Columns (62) have mixed types. Specify dtype option on import or set low_memory=False.
  self.all_data = pd.read_csv(cache_path)


[BirdAnalysis] All Complete.
12 BFs, 19 ZFs
BFs: ['g53o49' 'o40o79' 'o7' 'o70o85' 'p14y63' 'p1y44' 'p23y87' 'w66w67'
 'y4p74' 'y62p92' 'y64p94' 'y98p10']
ZFs: ['b45y83' 'b93o9' 'k35o47' 'k46o30' 'o13g88' 'o1o2' 'o27g88' 'o5o6'
 'o62g87' 'o68g47' 'p62b22' 'pi20k93' 'r100p13' 'r30p90' 'r87p67' 'w28o78'
 'w47o64' 'w85k85' 'y92y99']


# Probe Testing Export

In [ ]:
df = analysis.all_data.copy()
df = df.loc[df["Subject.Success"] == 1].copy()
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# force type to numeric
for col in ["Hit", "Miss", "Miss (NR)", "FA", "CR", "CR (NR)", "Timeout"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# filter for first 80 probe trials
probe_trial_max = 80
probe = df.loc[df['Block'].astype(str).isin(PROBE_ABC_VALID_BLOCKS)].copy()
probe["Probe Trial Count"] = probe.groupby(["Subject.Name", "Block ID"])["Probe Trials"].cumsum()
probe = probe.loc[probe["Probe Trial Count"] <= probe_trial_max]

groups = ['Subject.Name', 'Block ID', 'Tempo', 'Trial Type']
probe_analysis = analysis.run_analysis(probe, groups).reset_index()
probe_analysis = probe_analysis.loc[probe_analysis['Block Retry'] == 1].copy()
probe_analysis["Sound"] = probe_analysis['Block'].map(PROBE_SOUND_MAP)

probe_analysis.to_csv(paths.output / 'probe_stats_by_tempo.csv')

groups = ['Subject.Name', 'Block ID', 'Trial Type']
probe_analysis = analysis.run_analysis(probe, groups).reset_index()
probe_analysis = probe_analysis.loc[probe_analysis['Block Retry'] == 1].copy()
probe_analysis["Sound"] = probe_analysis['Block'].map(PROBE_SOUND_MAP)

probe_analysis.to_csv(paths.output / 'probe_stats.csv')

rt_probe = probe.loc[(probe["Block Retry"] == 1) & probe["RT"].notna()].copy()
rt_probe["Sound"] = rt_probe["Block"].map(PROBE_SOUND_MAP)

rt_probe.to_csv(paths.output / 'rt_probe.csv', index=False)

# Rule Training Export

In [10]:
GROUP_SIZE = 10
NUM_TRIALS = 1000

# (species, block, exclude_tempo, range_label)
RANGES = [
    ("ZF", "testing range 75-275d", 275, "75-275"),
    ("BF", "40-240d bengalese",     240, "40-240"),
    ("BF", "40-275d bengalese",     275, "40-275"),
]

# IOI edges: ZF (35,45,...,275), BF (40,50,...,280). Shared gap edges: (0,10,...,240).
IOI_BIN_EDGES = {
    "ZF": list(range(35, 276, GROUP_SIZE)),
    "BF": list(range(40, 281, GROUP_SIZE)),
}
GAP_BIN_EDGES = list(range(0, 241, GROUP_SIZE))

def _bin_center(series, edges):
    return pd.cut(series, bins=edges, right=False).map(
        lambda x: x.mid if pd.notna(x) else np.nan).astype(float)

frames = []
for species, block, exclude_tempo, range_label in RANGES:
    d = analysis.all_data.loc[analysis.all_data["Block"] == block].copy()
    if species == "BF" and range_label == "40-275":
        d = d.loc[d["Subject.Name"] != "w66w67"].copy()
    d = d.loc[d["Subject.Success"] == 1].copy()
    d = d.loc[d["Tempo"] != exclude_tempo]
    d = d.sort_values(["Subject.Name", "Date", "Time"])
    d = d.groupby("Subject.Name", group_keys=False).head(NUM_TRIALS)

    for c in ["Hit", "Miss", "Miss (NR)", "FA", "CR", "CR (NR)", "Total Hit", "Total CR"]:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0)

    d["Gap_ms"]       = d["Tempo"] - SYLLABLE_DUR_MS[species]
    d["Tempo Center"] = _bin_center(d["Tempo"],  IOI_BIN_EDGES[species])
    # Gap_Mid holds gap-bin CENTERS, not IOI-bin centers shifted by sound duration
    d["Gap_Mid"]      = _bin_center(d["Gap_ms"], GAP_BIN_EDGES)
    d["Correct"]      = ((d["Total Hit"] == 1) | (d["Total CR"] == 1)).astype(int)
    d["Species"]      = species
    d["RangeGroup"]   = range_label
    frames.append(d)

trial_df = pd.concat(frames, ignore_index=True)

# ── TRIAL-LEVEL EXPORT ──
keep = ["Subject.Name", "Species", "Subject.Sex", "RangeGroup", "Date", "Time", "Trial Num",
        "Tempo", "Tempo Center", "Gap_ms", "Gap_Mid", "Correct", "Total Hit", "Total Miss", "Total FA", "Total CR", "Hit", "Miss", "Miss (NR)", "FA", "CR", "CR (NR)", "Class", "Trial Type", "RT", "Timeout", "Stimulus"]
keep = [c for c in keep if c in trial_df.columns]
trial_df = trial_df[keep].sort_values(["Species", "Subject.Name", "Date", "Time"]).reset_index(drop=True)
trial_df.to_csv(paths.output / "rule_training_stats_by_trial.csv", index=False)
print(f"[trial]  {len(trial_df):,} trials, {trial_df['Subject.Name'].nunique()} birds")

bird_summary = (
    trial_df.groupby(["Species", "RangeGroup"], observed=True)
    .agg(
        n_birds=("Subject.Name", "nunique"),
        birds=("Subject.Name", lambda x: ", ".join(sorted(x.dropna().astype(str).unique()))),
    )
    .reset_index()
    .sort_values(["Species", "RangeGroup"])
)
print("\n===== BIRDS BY SPECIES x RANGE =====")
for species, species_df in bird_summary.groupby("Species", sort=True):
    print(f"{species} ({species_df['n_birds'].sum()} birds)")
    for _, row in species_df.iterrows():
        print(f"  {row['RangeGroup']}: n={row['n_birds']} | {row['birds']}")

# ── GROUP-LEVEL EXPORT: one row per Species x bird x bin  ──
#    Two csvs from the same trials, one by tempo and one by gap duration
SOUND_DUR = SYLLABLE_DUR_MS

def compute_metrics(g):
    hits   = g["Hit"].sum();  misses = g["Miss"].sum();  miss_nr = g["Miss (NR)"].sum()
    fas    = g["FA"].sum();   crs    = g["CR"].sum();     cr_nr   = g["CR (NR)"].sum()
    sig    = hits + misses + miss_nr          # all S+ trials
    noise  = crs + cr_nr + fas                # all S- trials
    hr = hits / sig if sig else np.nan
    fr = fas  / noise if noise else np.nan
    cr_rate = (crs + cr_nr) / noise if noise else np.nan
    # d' and criterion via the shared SignalDetectionStats log-linear
    # (half-count) correction, so this export matches the probe exports.
    stats_obj = SignalDetectionStats(
        n_hit=np.array([hits]),
        n_miss=np.array([misses + miss_nr]),
        n_fa=np.array([fas]),
        n_cr=np.array([crs + cr_nr]),
    )
    dprime, _, criterion = stats_obj.dprime_beta_criterion()
    return pd.Series({
        "n_trials": len(g), "Overall Acc": g["Correct"].mean(),
        "Hit Rate": hr, "FA Rate": fr, "CR Rate": cr_rate,
        "d Prime": dprime[0], "Criterion": criterion[0],
        "RT mean": g["RT"].mean() if "RT" in g.columns else np.nan,
    })

def group_export(bin_col, fname, label):
    gdf = (
        trial_df.dropna(subset=[bin_col])
        .groupby(["Species", "Subject.Name", bin_col], observed=True)
        .apply(compute_metrics, include_groups=False)
        .reset_index()
    )
    sd = gdf["Species"].map(SOUND_DUR)
    if bin_col == "Tempo Center":          # IOI bins -> add matching gap center
        gdf["Gap Center"] = gdf["Tempo Center"] - sd
    else:                                  # gap bins -> add matching IOI center
        gdf["IOI Center"] = gdf["Gap_Mid"] + sd
    gdf = gdf.sort_values(["Species", "Subject.Name", bin_col]).reset_index(drop=True)
    gdf.to_csv(paths.output / fname, index=False)
    print(f"[group:{label}]  {len(gdf):,} (bird x bin) rows -> {fname}")
    summ = (gdf.groupby(["Species", bin_col], observed=True)
            .agg(n_birds=("Subject.Name", "nunique"),
                 mean_acc=("Overall Acc", "mean"),
                 sem_acc=("Overall Acc", lambda x: x.std() / np.sqrt(len(x)) if len(x) > 1 else 0.0),
                 mean_dprime=("d Prime", "mean"))
            .reset_index())
    print(f"\n===== SPECIES x {label} SUMMARY =====")
    print(summ.to_string(index=False))

group_export("Gap_Mid",      "rule_training_by_gap_bin.csv", "GAP")
group_export("Tempo Center", "rule_training_by_ioi_bin.csv", "IOI")

[trial]  20,000 trials, 20 birds

===== BIRDS BY SPECIES x RANGE =====
BF (9 birds)
  40-240: n=2 | w66w67, y64p94
  40-275: n=7 | o7, o70o85, p14y63, p1y44, p23y87, y4p74, y98p10
ZF (11 birds)
  75-275: n=11 | k35o47, o13g88, o1o2, o27g88, o5o6, o62g87, o68g47, pi20k93, w47o64, w85k85, y92y99
[group:GAP]  430 (bird x bin) rows -> rule_training_by_gap_bin.csv

===== SPECIES x GAP SUMMARY =====
Species  Gap_Mid  n_birds  mean_acc  sem_acc  mean_dprime
     BF      5.0        9  0.471718 0.021684    -0.013949
     BF     15.0        9  0.453534 0.030352    -0.136009
     BF     25.0        9  0.576551 0.025446     0.522373
     BF     35.0        9  0.620373 0.046580     0.860813
     BF     45.0        9  0.688515 0.041644     1.341761
     BF     55.0        9  0.747423 0.044032     1.665839
     BF     65.0        9  0.734667 0.039464     1.644065
     BF     75.0        9  0.694952 0.034385     1.280895
     BF     85.0        9  0.697573 0.032256     1.252337
     BF     95.0       

In [9]:
# Run after Initialization. This cell can run without the earlier Rule Training Export cell.
# It replaces only rule_training_stats_by_trial.csv; grouped-bin CSVs are unchanged.
import numpy as np
import pandas as pd

ranges_1175 = [
    ("ZF", "testing range 75-275d", 275, "75-275"),
    ("BF", "40-240d bengalese", 240, "40-240"),
    ("BF", "40-275d bengalese", 275, "40-275"),
]
ioi_edges_1175 = {
    "ZF": list(range(35, 276, 10)),
    "BF": list(range(40, 281, 10)),
}
gap_edges_1175 = list(range(0, 241, 10))
sound_dur_1175 = {"ZF": 75, "BF": 38}


def bin_center_1175(series, edges):
    return pd.cut(series, bins=edges, right=False).map(
        lambda interval: interval.mid if pd.notna(interval) else np.nan
    ).astype(float)


frames_1175 = []
for species, block, exclude_tempo, range_label in ranges_1175:
    d = analysis.all_data.loc[analysis.all_data["Block"] == block].copy()
    if species == "BF" and range_label == "40-275":
        d = d.loc[d["Subject.Name"] != "w66w67"].copy()
    d = d.loc[d["Subject.Success"] == 1].copy()
    d = d.loc[d["Tempo"] != exclude_tempo]
    d = d.sort_values(["Subject.Name", "Date", "Time"])
    trial_limit = 1175 if species == "BF" and range_label == "40-275" else 1000
    d = d.groupby("Subject.Name", group_keys=False).head(trial_limit)

    for column in ["Hit", "Miss", "Miss (NR)", "FA", "CR", "CR (NR)",
                   "Total Hit", "Total CR"]:
        if column in d.columns:
            d[column] = pd.to_numeric(d[column], errors="coerce").fillna(0)

    d["Gap_ms"] = d["Tempo"] - sound_dur_1175[species]
    d["Tempo Center"] = bin_center_1175(d["Tempo"], ioi_edges_1175[species])
    d["Gap_Mid"] = bin_center_1175(d["Gap_ms"], gap_edges_1175)
    d["Correct"] = ((d["Total Hit"] == 1) | (d["Total CR"] == 1)).astype(int)
    d["Species"] = species
    d["RangeGroup"] = range_label
    frames_1175.append(d)

trial_df = pd.concat(frames_1175, ignore_index=True)
columns_1175 = [
    "Subject.Name", "Species", "Subject.Sex", "RangeGroup", "Date", "Time",
    "Trial Num", "Tempo", "Tempo Center", "Gap_ms", "Gap_Mid", "Correct",
    "Total Hit", "Total Miss", "Total FA", "Total CR", "Hit", "Miss",
    "Miss (NR)", "FA", "CR", "CR (NR)", "Class", "Trial Type", "RT",
    "Timeout", "Stimulus",
]
columns_1175 = [column for column in columns_1175 if column in trial_df.columns]
trial_df = trial_df[columns_1175].sort_values(
    ["Species", "Subject.Name", "Date", "Time"]
).reset_index(drop=True)
trial_df.to_csv(paths.output / "rule_training_stats_by_trial.csv", index=False)
print(f"[trial, 1175 version] {len(trial_df):,} trials, "
      f"{trial_df['Subject.Name'].nunique()} birds")
print(trial_df.groupby(["Species", "RangeGroup"])["Subject.Name"]
      .agg(trials="size", birds="nunique"))


[trial, 1175 version] 21,225 trials, 20 birds
                    trials  birds
Species RangeGroup               
BF      40-240        2000      2
        40-275        8225      7
ZF      75-275       11000     11


# Fig S1 Tempo vs CV / VRMS

In [ ]:
# CV panels show irregular stimuli only
PROBE_DIR = STIM_DIR / "Training_Probe"
RULE_DIR  = STIM_DIR / "Rule_Training_40-275"

SOUND_STYLE = {
    "Sound A": dict(marker="^", color="#FFDC98", label="CON1"),
    "Sound B": dict(marker="<", color="#FAB89A", label="CON2"),
    "Sound C": dict(marker=">", color="#C08B9B", label="CON3"),
}
RULE_KW = dict(s=42, alpha=0.45, edgecolors="none")
TRI_KW  = dict(s=95, alpha=0.9, edgecolors="black", linewidth=0.6)

def _rule_df(core_dir, species, prefix):
    """Extract one rule-stimulus directory into a reusable table."""
    df = analysis.extract_stimulus_info(
        species=species, core_dir=core_dir,
        stim_path=core_dir / "Stimulus Identification.txt",
        prefix_zf=prefix, vrms=True,
    )
    df = df[df['Block'] == f'{species} Rule'].copy()
    df["Actual Avg IOI (ms)"] = pd.to_numeric(df["Actual Avg IOI (ms)"], errors="coerce")
    df['CV'] = pd.to_numeric(df['CV'], errors='coerce')
    df['V_RMS'] = pd.to_numeric(df['V_RMS'], errors='coerce')
    return df


def _rule_xy(df, ycol, regularity=None):
    """Plot coordinates from a rule table"""
    if regularity is not None:
        df = df[df['Stimulus'].str.contains(regularity, case=False, na=False)]
    df = df.dropna(subset=['Actual Avg IOI (ms)', ycol])
    return df["Actual Avg IOI (ms)"].to_numpy(), df[ycol].to_numpy()


def _probe_df(df, species, regularity=None):
    """Select one species from the shared probe table"""
    df = df[df['Species'] == species].copy()
    if regularity is not None:
        df = df[df['Stimulus'].str.contains(regularity, case=False, na=False)]
    df["Actual Avg IOI (ms)"] = pd.to_numeric(df["Actual Avg IOI (ms)"], errors="coerce")
    df["CV"]    = pd.to_numeric(df["CV"], errors="coerce")
    df["V_RMS"] = pd.to_numeric(df["V_RMS"], errors="coerce")
    return df


# --- Extract data ---
zf_rule_data = _rule_df(RULE_DIR, 'ZF', 'z')
bf_rule_data = _rule_df(RULE_DIR, 'BF', 'g')
probe_data = analysis._extract_training_probe_info(PROBE_DIR)

# CV panels: irregular only
zf_rule_cv = _rule_xy(zf_rule_data, 'CV', 'ir')
bf_rule_cv = _rule_xy(bf_rule_data, 'CV', 'ir')
zf_probe_cv = _probe_df(probe_data, 'ZF', 'ir')
bf_probe_cv = _probe_df(probe_data, 'BF', 'ir')

# V_RMS panels: regular + irregular
zf_rule_rms = _rule_xy(zf_rule_data, 'V_RMS')
bf_rule_rms = _rule_xy(bf_rule_data, 'V_RMS')
zf_probe_rms = _probe_df(probe_data, 'ZF')
bf_probe_rms = _probe_df(probe_data, 'BF')


# --- Shared y-limits per column; per-species x-limits ---
def _cat(*arrs):
    arrs = [np.asarray(a, dtype=float) for a in arrs if len(a)]
    return np.concatenate(arrs) if arrs else np.array([0.0])

cv_all = _cat(zf_rule_cv[1], bf_rule_cv[1],
              zf_probe_cv["CV"].to_numpy(), bf_probe_cv["CV"].to_numpy())
rms_all = _cat(zf_rule_rms[1], bf_rule_rms[1],
               zf_probe_rms["V_RMS"].dropna().to_numpy(),
               bf_probe_rms["V_RMS"].dropna().to_numpy())

CV_LIM  = (0, float(np.nanmax(cv_all)) * 1.1)
RMS_LIM = (0, float(np.nanmax(rms_all)) * 1.1)

def _pad_x(lim, frac=0.03):
    """Pad both ends of an x-range"""
    lo, hi = lim
    pad = (hi - lo) * frac
    return (lo - pad, hi + pad)

X_LIMS  = {sp: _pad_x(lim) for sp, lim in [("ZF", (75, 275)), ("BF", (40, 275))]}

scatter_color = '#83BCBC'

def _plot_probe(ax, pdf, ycol):
    for snd, sty in SOUND_STYLE.items():
        g = pdf[pdf["Sound"] == snd].dropna(subset=[ycol])
        ax.scatter(g["Actual Avg IOI (ms)"], g[ycol],
                   marker=sty["marker"], color=sty["color"], label=sty["label"], **TRI_KW)


def _style(ax, xlim, ylim, ylabel):
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_xlabel("IOI (ms)", fontsize=12, fontfamily="Arial")
    ax.set_ylabel(ylabel, fontsize=12, fontfamily="Arial")
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=12)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontfamily("Arial")


# ----- Four separate panels, each exported as SVG and PNG -----
SPECIES_LABEL = {"ZF": "Zebra Finch", "BF": "Bengalese Finch"}

# (species, filename tag, rule data, probe data, y-column, y-limits, y-label)
panels = [
    ("ZF", "CV",    zf_rule_cv,  zf_probe_cv,  "CV",    CV_LIM,  "IOI CV"),
    ("ZF", "V_RMS", zf_rule_rms, zf_probe_rms, "V_RMS", RMS_LIM, r"$V_{\text{RMS}}$"),
    ("BF", "CV",    bf_rule_cv,  bf_probe_cv,  "CV",    CV_LIM,  "IOI CV"),
    ("BF", "V_RMS", bf_rule_rms, bf_probe_rms, "V_RMS", RMS_LIM, r"$V_{\text{RMS}}$"),
]

# Legend entries shared by all four panels
legend_handles = []
for snd, sty in SOUND_STYLE.items():
    legend_handles.append(
        plt.Line2D([], [], linestyle="none", marker=sty["marker"], markersize=9.7,
                   markerfacecolor=sty["color"], markeredgecolor="black",
                   markeredgewidth=0.6, alpha=0.9, label=sty["label"]))
legend_handles.append(
    plt.Line2D([], [], linestyle="none", marker="o", markersize=6.5,
               markerfacecolor=scatter_color, markeredgecolor="none",
               alpha=0.45, label="CON4"))

# Each panel is one column wide (84 mm) so the four assemble into a 2 x 2,
# double-column figure at 100 %. The shared legend is exported separately.
for sp, name, rule_xy, probe, ycol, ylim, ylabel in panels:
    fig, ax = plt.subplots(figsize=(FIG_COL1_IN, 2.9), layout="constrained")
    ax.scatter(*rule_xy, color=scatter_color, **RULE_KW)
    _plot_probe(ax, probe, ycol)
    _style(ax, X_LIMS[sp], ylim, ylabel)
    ax.set_title(SPECIES_LABEL[sp], fontsize=12, fontfamily="Arial", loc="center")
    save_figure(fig, paths.output / f"FigS1_{sp}_{name}")
    plt.show()

# One vertical legend for all four panels.
legend_fig, legend_ax = plt.subplots(figsize=(FIG_COL1_IN, 1.35))
legend_ax.axis("off")
legend_ax.legend(handles=legend_handles, ncol=1, loc="center",
                 frameon=False, prop={"family": "Arial", "size": 12}, handletextpad=0.4,
                 labelspacing=0.5)
save_figure(legend_fig, paths.output / "FigS1_Tempo_CV_VRMS_Legend")
plt.show()

# Fig S5 Rule CV Colored by Accuracy — Binned Tempi

In [ ]:
def plot_figS5_rule_cv_accuracy_binned(bin_width_ms=10):
    """Plot rule CV and accuracy on species-specific fixed IOI grids."""
    if not isinstance(bin_width_ms, (int, np.integer)) or bin_width_ms < 1:
        raise ValueError("bin_width_ms must be a positive integer")

    # Rebuild the selected trial and irregular-stimulus tables here so this
    # function does not depend on running the unbinned accuracy figure first.
    color_trials = trial_df.loc[
        trial_df["Species"].eq("ZF")
        | (trial_df["Species"].eq("BF") & trial_df["Subject.Sex"].eq("Male"))
    ].copy()
    color_trials["Tempo"] = pd.to_numeric(color_trials["Tempo"], errors="coerce")
    color_trials["Correct"] = pd.to_numeric(color_trials["Correct"], errors="coerce")
    color_trials = color_trials.dropna(subset=["Tempo", "Correct"])

    rule_points = pd.concat(
        [zf_rule_data.assign(Species="ZF"), bf_rule_data.assign(Species="BF")],
        ignore_index=True,
    )
    rule_points = rule_points.loc[
        rule_points["Stimulus"].str.contains("ir", case=False, na=False)
    ].copy()
    tempo_code = rule_points["Stimulus"].str.extract(
        r"_(\d{4})(?:ir1|reg)\.txt$", expand=False
    )
    rule_points["Tempo"] = pd.to_numeric(tempo_code, errors="coerce") / 10.0
    available_tempi = color_trials[["Species", "Tempo"]].drop_duplicates()
    rule_points = rule_points.merge(
        available_tempi, on=["Species", "Tempo"], how="inner", validate="many_to_one"
    )

    # Match Rule Training Export: ZF edges start at 35 ms; BF at 40 ms.
    bin_starts = {"ZF": 35.0, "BF": 40.0}
    species_bins = []
    for species, bin_start in bin_starts.items():
        species_points = rule_points.loc[rule_points["Species"].eq(species)].copy()
        max_tempo = max(
            species_points["Tempo"].max(),
            color_trials.loc[color_trials["Species"].eq(species), "Tempo"].max(),
        )
        bin_stop = np.nextafter(max_tempo + bin_width_ms, np.inf)
        bin_edges = np.arange(bin_start, bin_stop, bin_width_ms, dtype=float)
        species_points["Tempo Bin"] = pd.cut(
            species_points["Tempo"], bins=bin_edges, right=False
        )
        species_bins.append(species_points)
    points_with_bins = pd.concat(species_bins, ignore_index=True)
    points_with_bins = points_with_bins.dropna(subset=["Tempo Bin"])

    tempo_to_bin = (
        points_with_bins[["Species", "Tempo", "Tempo Bin"]]
        .drop_duplicates()
    )
    trials_with_bins = color_trials.merge(
        tempo_to_bin, on=["Species", "Tempo"], how="inner", validate="many_to_one"
    )

    # CV is averaged across rule points; accuracy is first calculated per bird
    # and then averaged equally across birds, matching the Fig3 mean curve.
    binned_points = (
        points_with_bins.groupby(["Species", "Tempo Bin"], observed=True)
        .agg(
            Mean_Actual_IOI=("Actual Avg IOI (ms)", "mean"),
            Plot_CV=("CV", "mean"),
            SD_CV=("CV", "std"),
            Tempo_Min=("Tempo", "min"),
            Tempo_Max=("Tempo", "max"),
            n_rule_points=("Stimulus", "size"),
        )
        .reset_index()
    )
    bird_accuracy = (
        trials_with_bins.groupby(
            ["Species", "Subject.Name", "Tempo Bin"], observed=True
        )
        .agg(
            Bird_Accuracy=("Correct", "mean"),
            n_trials=("Correct", "size"),
        )
        .reset_index()
    )
    bin_accuracy = (
        bird_accuracy.groupby(["Species", "Tempo Bin"], observed=True)
        .agg(
            Proportion_Correct=("Bird_Accuracy", "mean"),
            n_trials=("n_trials", "sum"),
            n_birds=("Subject.Name", "nunique"),
        )
        .reset_index()
    )
    binned_points = binned_points.merge(
        bin_accuracy, on=["Species", "Tempo Bin"], how="inner", validate="one_to_one"
    )
    binned_points["Bin_Center"] = (
        binned_points["Tempo Bin"].map(lambda interval: interval.mid).astype(float)
    )
    # With 1 ms bins, retain the exact original measured IOI coordinates.
    binned_points["Plot_IOI"] = (
        binned_points["Mean_Actual_IOI"]
        if bin_width_ms == 1 else binned_points["Bin_Center"]
    )
    binned_points.attrs["bin_width_ms"] = int(bin_width_ms)

    # Fixed grayscale: 0.4 is black and 1.0 is white, for every bin width.
    accuracy_norm = mpl.colors.Normalize(vmin=0.4, vmax=1.0, clip=True)

    # Match species_colors in Fig2.R; species are identified by the outlines.
    species_styles = {
        "ZF": dict(color="#D9A87E", marker="o"),
        "BF": dict(color="#7EB6D9", marker="o"),
    }
    cv_sd = binned_points["SD_CV"].fillna(0)
    cv_min = min(0.0, float((binned_points["Plot_CV"] - cv_sd).min()) * 1.1)
    cv_max = float((binned_points["Plot_CV"] + cv_sd).max()) * 1.1
    fig, ax = plt.subplots(figsize=(5, 3))
    fig.subplots_adjust(bottom=0.20)  # Keep the IOI label inside the uncropped export.
    for species, style in species_styles.items():
        points = binned_points.loc[binned_points["Species"].eq(species)]
        error_points = points.dropna(subset=["Plot_CV", "SD_CV"])
        if not error_points.empty:
            ax.errorbar(
                error_points["Plot_IOI"],
                error_points["Plot_CV"],
                yerr=error_points["SD_CV"],
                fmt="none",
                ecolor=style["color"],
                elinewidth=0.8,
                capsize=2,
                capthick=0.8,
                zorder=2,
            )
        scatter = ax.scatter(
            points["Plot_IOI"],
            points["Plot_CV"],
            c=points["Proportion_Correct"],
            cmap="gray",
            norm=accuracy_norm,
            s=30,
            marker=style["marker"],
            edgecolors=style["color"],
            linewidths=0.8,
            zorder=3,
        )

    ax.set_xlim(_pad_x((40, 275)))
    ax.set_ylim(cv_min, cv_max)
    ax.set_xlabel("IOI (ms)")
    ax.set_ylabel("Mean IOI CV")
    ax.set_facecolor("white")
    ax.spines[["top", "right"]].set_visible(False)
    colorbar = fig.colorbar(scatter, ax=ax, ticks=[0.4, 0.6, 0.8, 1.0], pad=0.03)
    colorbar.set_label("Proportion correct")
    fig.patch.set_facecolor("white")

    save_figure(
        fig, paths.output / f"FigS5_rule_CV_accuracy_binned_{bin_width_ms}ms"
    )
    plt.show()
    display(binned_points)
    return binned_points

In [ ]:
# Change this argument to choose the fixed IOI-bin width in milliseconds.
plot_figS5_rule_cv_accuracy_binned(bin_width_ms=10)

# Learning Curves

In [ ]:
analysis.plot_figS3_group(species='ZF')
analysis.plot_figS3_group(species='BF')
analysis.plot_discrim_learning_curve(species='BF')

In [ ]:
# Median birds from the pooled trials-to-criterion cell (BF, ZF)
analysis.plot_bird_learning_curve(["o68g47", "y98p10"])

# Trials to Criterion

In [ ]:
# ── Trials to criterion: one row per bird × sound (A/B/C)
ttc = analysis.compute_trials_to_criteria().reset_index()
ttc["Sound"] = ttc["Block"].map(TRAINING_ONLY_SOUND_MAP)
ttc = ttc[ttc["Sound"].notna() & ttc["Block Retry"].eq(1)]

ttc = ttc.merge(
    analysis.bird_stats[["Subject.Name", "Subject.Sex", "Subject.Order"]]
            .drop_duplicates("Subject.Name"),
    on="Subject.Name", how="left",
)

cols = ["Subject.Name", "Subject.Species", "Subject.Sex", "Subject.Order",
        "Sound", "Trials to Criteria", "Total_Trials", "Block Completed", "Start_Date"]
trials_to_criterion = (
    ttc[cols]
    .sort_values(["Subject.Species", "Subject.Name", "Sound"])
    .reset_index(drop=True)
)

trials_to_criterion.to_csv(paths.output / "trials_to_criterion.csv", index=False)

# Average Trials per day ZF vs. BF

In [ ]:
# Trial/day per bird, then mean/SD for each species.
td = analysis.all_data.assign(Date=pd.to_datetime(analysis.all_data["Date"], errors="coerce"))
trials_per_day = (
    td.groupby(["Subject.Species", "Subject.Name"], observed=True)
    .agg(Trials=("Trials", "size"), Days=("Date", "nunique"))
    .assign(Trials_per_Day=lambda x: x.Trials / x.Days)
    .groupby("Subject.Species", observed=True)["Trials_per_Day"]
    .agg(["mean", "std", "count"])
)
print(trials_per_day)

# Reaction Time: Last 500 Valid Responses per Bird before Probe Testing

In [ ]:
df = analysis.all_data.copy()

# Last 500 responses with an RT (no NR trials) per bird x sound, pooled over training and reduced reinforcement blocks
base_block = df["Block"].str.replace(" 0.8", "", regex=False)

rt_reduced_last500 = (
    df[base_block.isin(TRAINING_ONLY_SOUND_MAP)
       & (df["Block Retry"] == 1)
       & (df["Subject.Success"] == 1)
       & df["NR"].eq(0)
       & df["RT"].notna()]
    .assign(Sound=lambda d: d["Block"].str.replace(" 0.8", "", regex=False)
                              .map(TRAINING_ONLY_SOUND_MAP))
    .sort_values(["Subject.Name", "Sound", "Date", "Time"])
    .groupby(["Subject.Name", "Sound"])
    .tail(500)
)

rt_reduced_last500.to_csv(paths.output / 'rt_last500.csv', index=False)

# Fig2A Legend

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

sound_colors = {'A': '#FFDB99', 'B': '#FAB99B', 'C': '#C08A9B'}

handles = [
    Patch(facecolor=sound_colors['A'], edgecolor='black', label='Sound A'),
    Line2D([], [], color='black', lw=1.2, label='Hit Rate'),
    Patch(facecolor=sound_colors['B'], edgecolor='black', label='Sound B'),
    Line2D([], [], color='black', lw=1.2, ls=':', label='CR Rate'),
    Patch(facecolor=sound_colors['C'], edgecolor='black', label='Sound C'),
    Line2D([], [], color='black', lw=2.5, label='Overall Performance'),
]

fig, ax = plt.subplots(figsize=(9, 1.5))
ax.axis('off')
ax.legend(handles=handles, ncol=3, loc='center', fontsize=8,
          frameon=True, facecolor='white', edgecolor='#D0D0D0',
          handlelength=1, handletextpad=0.5, columnspacing=1.3,
          borderpad=0.6, labelspacing=0.4)
plt.show()
save_figure(fig, analysis.paths.output / "Fig2_legend")

# Tempo Discrim Export

In [ ]:
# Tempo discrimination: last 500 trials of the discrim block per bird, with S+ tempo group and sex
df = analysis.all_data[(analysis.all_data["Subject.Group"] == "Tempo Discrim")
                       & analysis.all_data["Block"].str.contains("discrim", na=False)]
df = df.sort_values(["Subject.Name", "Date", "Time"]).groupby("Subject.Name").tail(500)

perf = analysis.run_analysis(df, groups=["Subject.Name"]).reset_index()
info = df.groupby("Subject.Name").agg(
    Species=("Subject.Species", "first"), Sex=("Subject.Sex", "first"), Block=("Block", "first")).reset_index()
info["Group"] = info["Block"].str.contains("144:120").map({True: "144 S+", False: "120 S+"})

out = perf.merge(info, on="Subject.Name")[["Subject.Name", "Species", "Sex", "Group",
                                            "Total Hit", "Total Miss", "Total FA", "Total CR"]]
out["n_correct"] = out["Total Hit"] + out["Total CR"]
out["n_trials"] = out[["Total Hit", "Total Miss", "Total FA", "Total CR"]].sum(axis=1)
out.to_csv(paths.output / "tempo_discrim_last500.csv", index=False)
print(out)